In [ ]:
import os, glob, subprocess
z = glob.glob('/kaggle/input/basma-code/*.zip')
if not os.path.exists('/kaggle/working/Basma'):
    if z: subprocess.run(['unzip', '-q', '-o', z[0], '-d', '/kaggle/working/Basma'], check=True)
    else: subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/us3ph/Basma.git', '/kaggle/working/Basma'], check=True)
os.chdir('/kaggle/working/Basma'); print(sorted(os.listdir('.')))

In [ ]:
import os, glob, json, subprocess
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], capture_output=True, text=True).stdout)
# data-kernel outputs (mounted at /kaggle/input/notebooks/<owner>/<slug>/): their manifests identify them
mans = sorted(m for m in glob.glob('/kaggle/input/**/manifest_*.jsonl', recursive=True) if '/basma-data-' in m)
dirs = sorted({os.path.dirname(m) for m in mans}); print('DATA DIRS', dirs)
assert len(dirs) == 4, f'expected the cf/recent/reals/eval builds, found {mans}'
# baseline scores from the first run: same shards and per-image attack seeds, so only new rows are scored
cache = sorted(glob.glob('/kaggle/input/**/runs/kaggle_base/baselines/thermostatic.jsonl', recursive=True))
os.environ['BASELINE_CACHE'] = os.path.dirname(cache[0]) if cache else ''; print('BASELINE_CACHE', os.environ['BASELINE_CACHE'] or 'none')
cfg = json.load(open('configs/kaggle_2xT4_v2.json')); cfg['data_dirs'] = dirs
json.dump(cfg, open('configs/kaggle_run.json', 'w'), indent=1); print(json.dumps(cfg, indent=1))

In [ ]:
# BASELINE_CACHE is in os.environ, which the shell inherits
!bash scripts/kaggle_train.sh configs/kaggle_run.json

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open('docs/RESULTS.md').read()))